In [1]:
import numpy as np

# ── 1. Define the HMM Parameters ─────────────────────────────────────────────
# Problem: Predict weather (hidden) from umbrella observation (visible)

states       = ['Sunny', 'Rainy']           # Hidden states
observations = ['No Umbrella', 'Umbrella']  # What we actually observe

# Start probabilities: which state are we likely in at day 1?
start_prob = np.array([0.6, 0.4])          # 60% Sunny, 40% Rainy

# Transition matrix: P(next state | current state)
# Row = current state, Col = next state
trans = np.array([[0.7, 0.3],              # Sunny → Sunny:0.7, Rainy:0.3
                  [0.4, 0.6]])             # Rainy → Sunny:0.4, Rainy:0.6

# Emission matrix: P(observation | hidden state)
# Row = hidden state, Col = observation
emit  = np.array([[0.8, 0.2],             # Sunny → No Umb:0.8, Umb:0.2
                  [0.3, 0.7]])             # Rainy → No Umb:0.3, Umb:0.7

# ── 2. Viterbi Algorithm — Find Most Likely Hidden State Sequence ─────────────
# Given observations, what sequence of hidden states best explains them?
def viterbi(obs_seq):
    T  = len(obs_seq)                      # Number of time steps
    N  = len(states)                       # Number of hidden states
    dp = np.zeros((T, N))                  # dp[t][s] = best prob at time t in state s
    bp = np.zeros((T, N), dtype=int)       # backpointer to trace best path

    # Initialization: start prob × emission prob for first observation
    dp[0] = start_prob * emit[:, obs_seq[0]]

    # Recursion: fill table step by step
    for t in range(1, T):
        for s in range(N):
            # For each state s, find which previous state gives max probability
            probs      = dp[t-1] * trans[:, s] * emit[s, obs_seq[t]]
            bp[t][s]   = np.argmax(probs)  # Remember best previous state
            dp[t][s]   = np.max(probs)

    # Backtrack: trace the best path from end to start
    path = np.zeros(T, dtype=int)
    path[-1] = np.argmax(dp[-1])           # Best final state
    for t in range(T-2, -1, -1):
        path[t] = bp[t+1][path[t+1]]

    return path, dp[-1].max()

# ── 3. Run Prediction ─────────────────────────────────────────────────────────
# Observed sequence: Umbrella, Umbrella, No Umbrella, Umbrella
# 0 = No Umbrella, 1 = Umbrella
obs_seq = [1, 1, 0, 1]

path, prob = viterbi(obs_seq)

print("Observations  :", [observations[o] for o in obs_seq])
print("Predicted States:", [states[s] for s in path])
print(f"Best Path Prob : {prob:.6f}")

Observations  : ['Umbrella', 'Umbrella', 'No Umbrella', 'Umbrella']
Predicted States: ['Rainy', 'Rainy', 'Rainy', 'Rainy']
Best Path Prob : 0.008891
